# Task 2: Reproducing UPM Algorithm Results

Aaron Satish


---

## Paper Information

**Title:** A self-verifying clustering approach to unsupervised matching of product titles
**Authors:** Leonidas Akritidis, Athanasios Fevgas, Panayiotis Bozanis, Christos Makris
**Journal:** Artificial Intelligence Review, Vol. 53, pp. 4777-4820 (2020)
**DOI:** https://link.springer.com/article/10.1007/s10462-020-09807-8

Page numbers below are the journal page numbers printed in the paper.

---

## Executive Summary

This notebook reproduces **UPM (Unsupervised Product Matcher)** from Akritidis et al. (2020) on the PriceRunner *aggregate* dataset. The algorithm clusters product listings using only their titles and vendor IDs:

1. **Stage 1, title analysis** (pp. 4782-4787): tokenization, token semantics, token and combination lexicons.
2. **Stage 2, initial clusters** (pp. 4787-4790): every title joins its highest-scoring token combination (Algorithm 2).
3. **Stage 3, verification** (pp. 4791-4796): no cluster may hold two products of one vendor (Algorithm 4).

**Main results**
- Paper F1 (p. 4811): **0.588**
- Reproduction F1, original (label-sorted) row order: **0.5798** (-0.0082)
- Reproduction F1, shuffled rows (3 seeds): **0.5752 +/- 0.0010** (-0.0128, -2.2% relative)


---

In [1]:
# Import required libraries
import time, json, os, math, re, itertools
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt

NOTEBOOK_START = time.perf_counter()
print("Libraries loaded successfully")
print("Versions: python-side libraries -> pandas", pd.__version__, "| numpy", np.__version__)

Libraries loaded successfully
Versions: python-side libraries -> pandas 3.0.5 | numpy 2.5.3


---
## Part 1: Algorithm Parameters

**Paper reference:** Sect. 4.2, pp. 4798-4801 (hyper-parameter setting and sensitivity study)

| Parameter | Symbol | Value | Paper reference |
|-----------|--------|-------|-----------------|
| **Max combination size** | K | 3 | Eq. 8 (p. 4800): K* = max(3, floor(l_t / 2) - 1), with mean title length l_t = 8.560 (Table 3) this gives max(3, 4 - 1) = 3. Checked against this data below. |
| **Distance smoothing** | alpha | 1.0 | Eq. 5; Fig. 3 right (p. 4799) |
| **Length normalization** | b | 1.0 | Eq. 6; Fig. 4 left (p. 4800) |
| **Similarity threshold** | tau | 0.5 | Fig. 4 right (p. 4800) |
| **Field weight** | Q(z) | 1 / abs(z) | Eq. 9 (p. 4801) |
| **Similarity measure** | - | IDF-weighted cosine | Table 4 (p. 4808); text p. 4802 |

**Implementation choices that the paper does not fix** (all listed again in Part 9):
- **Tie rule:** strict `>`, as in Algorithm 2, line 16 (p. 4789). `>=` is tested as an ablation.
- **Log base:** the paper writes log without a base; base 10 is used. A different base multiplies every idf and every score by one constant, so all rankings and cosine thresholds are unchanged.
- **Measurement-unit list:** the paper describes the lexicon (p. 4783) but does not list it; the list comes from the authors' code.
- **Small clusters:** for clusters of at most two products the clustroid is the longest title.
- **Titles without a positive-scoring combination** (one-token titles, or titles whose combinations all occur once): singleton cluster.
- **Token roles:** the semantics stored when a token is first seen in the dataset.
- **Vendor visiting order** in Algorithm 4: ascending vendor ID.

In [2]:
# Algorithm parameters (pp. 4799-4801)
K = 3           # Maximum combination size 
ALPHA = 1.0     # Distance smoothing 
B = 1.0         # Length normalisation 
TAU = 0.5       # Verification similarity threshold 

print("Algorithm Parameters (from paper Sect. 4.2):")
print(f"  K (max combination size): {K}")
print(f"  alpha (distance smoothing): {ALPHA}")
print(f"  b (length normalization): {B}")
print(f"  tau (similarity threshold): {TAU}")
print("  IDF / frequency log base: 10 (any base gives the same ranking, see text)")

Algorithm Parameters (from paper Sect. 4.2):
  K (max combination size): 3
  alpha (distance smoothing): 1.0
  b (length normalization): 1.0
  tau (similarity threshold): 0.5
  IDF / frequency log base: 10 (any base gives the same ranking, see text)


---
## Part 2: Load and Inspect Dataset

**Paper reference:** Sect. 4.1 and Table 3 (p. 4798)

In [3]:
# Load the PriceRunner aggregate dataset
df = pd.read_csv("pricerunner_aggregate.csv")
df.columns = df.columns.str.strip()

print("Dataset statistics (cf. Table 3, p. 4798):")
print(f"  Titles in CSV: {len(df):,}   | paper: 35,305   | difference: {len(df) - 35305}")
print(f"  Ground-truth clusters in CSV: {df['Cluster ID'].nunique():,}   | paper |P| (products): 15,844")
print(f"  Merchants: {df['Merchant ID'].nunique()}   | paper |V|: 306")
print(f"\nCSV is sorted by Cluster ID: {df['Cluster ID'].is_monotonic_increasing}")
print("   (the file order follows the ground truth, so we also run on shuffled rows)")

print("\nFirst 5 products:")
display(df.head())

Dataset statistics (cf. Table 3, p. 4798):
  Titles in CSV: 35,311   | paper: 35,305   | difference: 6
  Ground-truth clusters in CSV: 13,233   | paper |P| (products): 15,844
  Merchants: 306   | paper |V|: 306

CSV is sorted by Cluster ID: True
   (the file order follows the ground truth, so we also run on shuffled rows)

First 5 products:


,Product ID,Product Title,Merchant ID,Cluster ID,Cluster Label,Category ID,Category Label
0,1,apple iphone 8 plus 64gb silver,1,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
1,2,apple iphone 8 plus 64 gb spacegrau,2,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
2,3,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...,3,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
3,4,apple iphone 8 plus 64gb space grey,4,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
4,5,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,5,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones


**Important:** `Cluster ID` is the ground truth and is used **only for evaluation** (Sect. 4.3, p. 4801). It is **not** an input of the algorithm. The algorithm uses only:
- **Product Title** (clustering)
- **Merchant ID** (verification stage, blocking condition of Eq. 7, p. 4791)

Dataset differences to the paper (Table 3, aggregate row): 35,305 titles vs 35,311 here, and 15,844 products vs 13,233 ground-truth clusters here. Vendors match (306). The CSV is therefore not identical to the data behind the paper.

---
## Part 3: Algorithm Implementation

**Paper reference:** Sect. 3, pp. 4781-4796

### Stage 1: Tokenization and semantic labeling
**Sect. 3.2.1 (pp. 4782-4784)** - lowercase, punctuation rule, duplicate removal, number + unit concatenation, token semantics (Table 1, p. 4783).

### Stage 2: Combination scoring and initial clusters
**Sect. 3.3 (pp. 4787-4790)** - k-combinations for k = 2..K (Algorithm 1, p. 4786); score I(c) of Eq. 5 with Y_c of Eq. 6 (pp. 4788-4789); each title joins its highest-scoring combination (Algorithm 2, p. 4789).

### Stage 3: Verification and refinement
**Sect. 3.4 (pp. 4791-4796)** - blocking condition (Eq. 7, p. 4791); clustroid, deletions, transfers and new clusters (Algorithm 4, p. 4792; text pp. 4792-4793).

In [4]:
# ============================================================================
# STAGE 1: TOKENIZATION (Sect. 3.2.1, pp. 4782-4784)
# ============================================================================

# Measurement units. The paper describes the lexicon (p. 4783) but does not list it;
# this list (prefix x unit) is taken from the authors' code.
UNITS = {prefix + unit
         for prefix in ["m", "n", "", "k", "g", "t"]
         for unit in ["b", "hz", "bps", "'", "m", "btu"]}

NUM = re.compile(r"[\d.,]+")

def tokenize(title, sep="digits", keep_hyphen=False):
    """
    Tokenize a product title following Sect. 3.2.1 (pp. 4782-4784).

    1. Case folding.
    2. Punctuation removal. The paper keeps only dots and commas that are thousands/decimal
       separators, so by default (sep="digits") a dot/comma stays inside a token only
       BETWEEN DIGITS ("5.5", "3,2"). sep="any" reproduces the earlier behaviour (between any
       alphanumerics) and is used as an ablation.
    3. Duplicate token removal (listed as a morphological step by the paper).
    4. Number + measurement unit are concatenated ("64" + "gb" -> "64gb", Table 1, p. 4783).
    5. Duplicates are removed once more, so "64gb" + "64 gb" cannot leave "64gb" twice
       (my interpretation: the paper wants no repeated tokens in a title).

    Hyphens and slashes: the paper says punctuation is removed apart from dots/commas that are
    separators and "hyphens and slashes that delimit tokens". This is ambiguous. By default they
    split tokens (keep_hyphen=False). keep_hyphen=True keeps them inside tokens and is an ablation.
    The nVidia example on p. 4782 treats "V335-001R" as one token.
    """
    title = title.lower()
    dot = r"(?<=\d)[.,](?=\d)" if sep == "digits" else r"[.,]"
    extra = r"|[\-/]" if keep_hyphen else ""
    pattern = r"[a-z0-9]+(?:(?:" + dot + extra + r")[a-z0-9]+)*"
    toks = list(dict.fromkeys(re.findall(pattern, title)))

    out, i = [], 0
    while i < len(toks):
        if NUM.fullmatch(toks[i]) and i + 1 < len(toks) and toks[i + 1] in UNITS:
            out.append(toks[i] + toks[i + 1])
            i += 2
        else:
            out.append(toks[i])
            i += 1
    return list(dict.fromkeys(out))

def classify_token_role(token, mixed_seen):
    """
    Token semantics of Table 1 (p. 4783); each role is one of the five virtual fields z1..z5:
      attribute      type 1: number + measurement unit ("64gb", "3.2ghz")
      first_model    type 2: first mixed (letters+digits) token that is not an attribute
      other_model    type 3: the remaining mixed tokens
      numeric_model  type 4: a number not followed by a unit
      normal         type 5: everything else
    """
    if any(re.fullmatch(r"[\d.,]+" + re.escape(u), token) for u in UNITS):
        return "attribute", mixed_seen
    if NUM.fullmatch(token):
        return "numeric_model", mixed_seen
    if re.search(r"[a-z]", token) and re.search(r"\d", token):
        role = "first_model" if mixed_seen == 0 else "other_model"
        return role, mixed_seen + 1
    return "normal", mixed_seen

# Test tokenization
test_title = "Apple iPhone 8 Plus 64GB Silver"
tokens = tokenize(test_title)
print(f"Example: '{test_title}'")
print(f"Tokens: {tokens}")
mixed = 0
print("\nToken roles (cf. Table 1, p. 4783):")
for tok in tokens:
    role, mixed = classify_token_role(tok, mixed)
    print(f"  {tok:12s} -> {role}")

print("\nSeparator rule check:")
for t in ["nvidia,geforce gtx1050 5.5 3,2 ghz", "apple mq8n2b/a 64 gb 64gb"]:
    print(f"  {t!r} -> {tokenize(t)}")

Example: 'Apple iPhone 8 Plus 64GB Silver'
Tokens: ['apple', 'iphone', '8', 'plus', '64gb', 'silver']

Token roles (cf. Table 1, p. 4783):
  apple        -> normal
  iphone       -> normal
  8            -> numeric_model
  plus         -> normal
  64gb         -> attribute
  silver       -> normal

Separator rule check:
  'nvidia,geforce gtx1050 5.5 3,2 ghz' -> ['nvidia', 'geforce', 'gtx1050', '5.5', '3,2ghz']
  'apple mq8n2b/a 64 gb 64gb' -> ['apple', 'mq8n2b', 'a', '64gb']


### K check (Eq. 8)
The value of K is derived from the mean title length, so it is verified on this data instead of being taken for granted.

In [5]:
# Check K against Eq. 8 (p. 4800): K* = max(3, floor(mean_tokens / 2) - 1)
mean_tokens = float(np.mean([len(tokenize(t)) for t in df["Product Title"]]))
K_star = max(3, int(mean_tokens // 2) - 1)
print(f"Mean tokens per title (this tokenizer): {mean_tokens:.3f}   | paper Table 3: 8.560")
print(f"K* from Eq. 8: {K_star}   (the paper's 8.560 also gives max(3, 4 - 1) = 3)")
assert K == K_star, "K does not match Eq. 8"

Mean tokens per title (this tokenizer): 8.479   | paper Table 3: 8.560
K* from Eq. 8: 3   (the paper's 8.560 also gives max(3, 4 - 1) = 3)


In [6]:
def run_upm(df, K=3, alpha=1.0, b=1.0, tau=0.5, shuffle_seed=None,
            tie="gt", avg_len="occurrence", keep_hyphen=False, sep="digits"):
    """
    Run the complete UPM algorithm (pp. 4781-4796).

    """
    t0 = time.perf_counter()

    d = df.reset_index(drop=True)
    if shuffle_seed is not None:
        d = d.sample(frac=1.0, random_state=shuffle_seed).reset_index(drop=True)

    n = len(d)
    merchants = d["Merchant ID"].astype(int).to_numpy()
    toks = [tokenize(t, sep=sep, keep_hyphen=keep_hyphen) for t in d["Product Title"]]

    # ---- Stage 1: token lexicon (Algorithm 1, p. 4786) ---------------------
    # Semantics are stored when a token is first inserted into the lexicon.
    sem = {}
    for ts in toks:
        ms = 0
        for t in ts:
            r, ms = classify_token_role(t, ms)
            sem.setdefault(t, r)

    # idf(w) = log(|P| / f_w) (Eq. 6, p. 4789). The log base is not given in the paper; base 10 is
    # used here. Changing the base multiplies every idf, hence every score and every cosine, by
    # the same constant, so no ranking and no threshold decision changes.
    tf = Counter(t for ts in toks for t in set(ts))
    idf = {t: math.log10(n / c) for t, c in tf.items()}

    # ---- Stage 1: combination lexicon (Algorithm 1) ------------------------
    freq = Counter()           # f_c
    dsum = defaultdict(float)  # distance accumulator d_c^acc

    for ts in toks:
        for k in range(2, min(K, len(ts)) + 1):
            for pos in itertools.combinations(range(len(ts)), k):
                key = tuple(sorted(ts[p] for p in pos))   # order-independent signature
                freq[key] += 1
                # Eq. 3 (p. 4788): d^2(c,t) = sum_w (o_w^(c) - o_w^(t))^2 ; position inside c is 0,1,..,k-1
                dsum[key] += math.sqrt(sum((p - j) ** 2 for j, p in enumerate(pos)))

    # Average length (in tokens) of all combinations in the dataset (below Eq. 6)
    if avg_len == "occurrence":
        L = sum(len(k) * f for k, f in freq.items()) / sum(freq.values())
    else:
        L = sum(len(k) for k in freq) / len(freq)

    # Eq. 4 (p. 4788): average distance
    avg_dist = {k: dsum[k] / freq[k] for k in freq}

    # ---- Stage 2: score combinations (Eq. 5-6, Algorithm 2) ----------------
    score_cache = {}
    def score(key):
        """
        I(c) = k * Y_c^2 / (alpha + d(c)) * log f_c                      (Eq. 5, p. 4788)
        Y_c  = sum_w idf(w) * Q(z_sw) / (1 - b + b * k / L)              (Eq. 6, p. 4789)
        Q(z) = 1 / |z|  (number of tokens of c that share the field)     (Eq. 9, p. 4801)
        """
        if key in score_cache:
            return score_cache[key]
        k = len(key)
        rc = Counter(sem[t] for t in key)
        y = sum(idf[t] / rc[sem[t]] for t in key) / ((1 - b) + b * k / L)
        s = (k / (alpha + avg_dist[key])) * y * y * math.log10(freq[key])
        score_cache[key] = s
        return s

    # Algorithm 2: the combination with the maximum score is the dominating cluster.
    # Line 16 is a strict '>' with max initialised to 0, so a title whose combinations all
    # score 0 (every combination occurs only once, f_c = 1) gets no cluster. We give such titles
    # (and one-token titles, which have no combination at all) a singleton cluster at the end.
    init = defaultdict(list)
    singles = []
    for i, ts in enumerate(toks):
        best, bs = None, 0.0
        for k in range(2, min(K, len(ts)) + 1):
            for c in itertools.combinations(ts, k):
                key = tuple(sorted(c))
                s = score(key)
                if (s > bs) if tie == "gt" else (s >= bs):
                    bs, best = s, key
        if best is None:
            singles.append(i)
        else:
            init[best].append(i)
    n_init = len(init)

    # ---- IDF-weighted cosine vectors (weighted cosine, Table 4 p. 4808) ---
    col = {t: j for j, t in enumerate(idf)}
    r_, c_, v_ = [], [], []
    for i, ts in enumerate(toks):
        for t in ts:
            r_.append(i); c_.append(col[t]); v_.append(idf[t])
    V = normalize(csr_matrix((v_, (r_, c_)), shape=(n, len(col))), norm="l2")  # dot product = cosine

    # ---- Stage 3a: clustroids and deletions (Algorithm 4, steps 1-20, p. 4792) ----
    active, evicted = [], []
    for label, members in init.items():
        # Clustroid: the product with the greatest total similarity to the others (Sect. 3.4.2).
        # For clusters of <=2 products the paper gives no rule (both tie); we take the longest title.
        if len(members) <= 2:
            rep = max(members, key=lambda i: len(toks[i]))
        else:
            S = (V[members] @ V[members].T).toarray()
            np.fill_diagonal(S, 0.0)
            rep = members[int(np.argmax(S.sum(axis=1)))]

        # Per vendor keep the product most similar to the clustroid (steps 9-16). The clustroid has
        # similarity 1 with itself, so it always stays; we enforce that explicitly.
        by_m = defaultdict(list)
        for i in members:
            by_m[merchants[i]].append(i)

        kept = []
        for m, ids in by_m.items():
            if rep in ids:
                keep = rep
            elif len(ids) == 1:
                keep = ids[0]
            else:
                sims = (V[ids] @ V[rep].T).toarray().ravel()
                keep = ids[int(np.argmax(sims))]
            kept.append(keep)
            evicted.extend(i for i in ids if i != keep)

        active.append({"members": kept, "rep": rep, "merchants": {merchants[i] for i in kept}})

    n_evicted = len(evicted)

    # ---- Stage 3b: move deleted products to the most similar cluster (steps 21-41) ----
    # Inverted logic of the paper: for each cluster, look for the most similar evicted product
    # per vendor, instead of searching a cluster for each evicted product (p. 4793).
    by_tok = defaultdict(list)
    for i in evicted:
        for t in set(toks[i]):
            by_tok[t].append(i)

    remaining = set(evicted)
    for c in active:
        pool = set()
        for t in set(toks[c["rep"]]):
            pool.update(by_tok.get(t, []))   # cosine is 0 without a shared token, so this is exact

        cand = defaultdict(list)
        for i in sorted(pool):
            if i in remaining and merchants[i] not in c["merchants"]:
                cand[merchants[i]].append(i)

        ci = sorted(i for l in cand.values() for i in l)
        if not ci:
            continue
        sims = dict(zip(ci, (V[ci] @ V[c["rep"]].T).toarray().ravel()))

        # The paper leaves the vendor visiting order open; we use ascending merchant ID.
        for m in sorted(cand):
            bi, bsim = None, tau        # strict '>' tau (step 30)
            for i in cand[m]:
                if sims[i] > bsim:
                    bsim, bi = sims[i], i
            if bi is not None:
                c["members"].append(bi)
                c["merchants"].add(m)
                remaining.discard(bi)

    n_reattached = n_evicted - len(remaining)

    # ---- Stage 3c: remaining products form / join new clusters U' (steps 42-56) ----
    new = []
    idx = defaultdict(set)
    for i in sorted(remaining, key=lambda i: (merchants[i], i)):
        m = merchants[i]
        cands = sorted({cid for t in set(toks[i]) for cid in idx[t]})
        cands = [cid for cid in cands if m not in new[cid]["merchants"]]

        best, bsim = None, tau
        if cands:
            sims = (V[[new[cid]["rep"] for cid in cands]] @ V[i].T).toarray().ravel()
            for cid, s in zip(cands, sims):
                if s > bsim:
                    bsim, best = s, cid

        if best is not None:
            new[best]["members"].append(i)
            new[best]["merchants"].add(m)
        else:
            new.append({"members": [i], "rep": i, "merchants": {m}})   # clustroid of a new cluster = first product
            for t in set(toks[i]):
                idx[t].add(len(new) - 1)

    # ---- Finalise ----------------------------------------------------------
    clusters = active + new
    for i in singles:   # implementation choice, not in the paper
        clusters.append({"members": [i], "rep": i, "merchants": {merchants[i]}})

    # Invariants
    assigned = [i for c in clusters for i in c["members"]]
    assert len(assigned) == len(set(assigned)) == n, "every row must be in exactly one cluster"
    assert all(len(c["merchants"]) == len(c["members"]) for c in clusters), "vendor rule (Eq. 7) violated"
    assert all(c["rep"] in c["members"] for c in clusters), "a clustroid left its cluster"

    pred = np.empty(n, dtype=int)
    for cid, c in enumerate(clusters):
        pred[c["members"]] = cid

    d["Predicted Cluster"] = pred
    d["n_tokens"] = [len(t) for t in toks]

    stats = {
        "rows": n,
        "init_clusters": n_init,
        "evicted": n_evicted,
        "reattached": n_reattached,
        "new_clusters": len(new),
        "final_clusters": len(clusters),
        "one_token_rows": int((d["n_tokens"] < 2).sum()),
        "no_positive_combination_rows": len(singles),
        "mean_tokens": float(np.mean(d["n_tokens"])),
        "avg_comb_len": L,
        "seconds": time.perf_counter() - t0,
    }
    return d, stats

print("UPM algorithm implementation complete")

UPM algorithm implementation complete


---
## Part 4: Evaluation Function

**Paper reference:** Sect. 4.3 (p. 4801)

The paper creates a pair record for every two products inside a predicted cluster and compares these pairs with the ground-truth "matches" file:

- **True positives (TP):** pairs that are in the same predicted cluster and in the same ground-truth cluster
- **Precision:** P = TP / predicted pairs
- **Recall:** R = TP / ground-truth pairs
- **F1:** F1 = 2PR / (P + R)  (Sect. 4.2, p. 4798)

In [7]:
def count_pairs(sizes):
    """Count unordered pairs: sum of n(n-1)/2 over groups."""
    s = np.asarray(sizes, dtype=np.int64)
    return int((s * (s - 1) // 2).sum())

def evaluate(d, drop_single_token=False):
    """
    Pairwise precision / recall / F1 (p. 4801): every pair of products inside a
    predicted cluster is compared with the pairs of the ground-truth ("matches") file.

    drop_single_token : evaluate only the titles with >= 2 tokens (35,305 rows, the paper's title count)
    """
    x = d[d["n_tokens"] >= 2] if drop_single_token else d

    tp = count_pairs(x.groupby(["Cluster ID", "Predicted Cluster"]).size())  # pairs in both
    pp = count_pairs(x.groupby("Predicted Cluster").size())                   # predicted pairs
    ap = count_pairs(x.groupby("Cluster ID").size())                          # ground-truth pairs

    p = tp / pp if pp else 0
    r = tp / ap if ap else 0
    f1 = 2 * p * r / (p + r) if p + r else 0
    return {"tp": tp, "precision": p, "recall": r, "f1": f1, "rows": len(x)}

print("Evaluation function ready (pairwise F1)")

Evaluation function ready (pairwise F1)


---
## Part 5: Run Main Experiment

**Paper reference:** Sect. 4.3.2 (p. 4811), PriceRunner aggregate, F1 = 0.588

### Experiment 1: original (sorted) row order

In [8]:
PAPER_F1 = 0.588   # Sect. 4.3.2, p. 4811 (Fig. 6c and Sect. 4.3.1 round it to 0.59)

print("=" * 80)
print("RUNNING UPM ALGORITHM ON ORIGINAL (SORTED) DATA")
print("=" * 80)

result_df, stats = run_upm(df, K=K, alpha=ALPHA, b=B, tau=TAU)
metrics = evaluate(result_df)
metrics_35305 = evaluate(result_df, drop_single_token=True)

print(f"\n{'Pipeline statistics:':<40}")
print(f"{'  Products processed:':<40} {stats['rows']:,}")
print(f"{'  Initial clusters':<40} {stats['init_clusters']:,}")
print(f"{'  Products evicted':<40} {stats['evicted']:,}")
print(f"{'    -> reattached to existing:':<40} {stats['reattached']:,}")
print(f"{'    -> new clusters formed:':<40} {stats['new_clusters']:,}")
print(f"{'  Final clusters:':<40} {stats['final_clusters']:,}")
print(f"{'  One-token titles:':<40} {stats['one_token_rows']}")
print(f"{'  Titles with no positive-score combination:':<40} {stats['no_positive_combination_rows']} (incl. the one-token titles; kept as singletons)")
print(f"{'  Mean tokens per title:':<40} {stats['mean_tokens']:.3f} (Table 3: 8.560)")
print(f"{'  Runtime:':<40} {stats['seconds']:.1f} seconds (paper: 11 s in C++, Table 5)")

print(f"\n{'Evaluation results:':<40}")
print(f"{'  Precision:':<40} {metrics['precision']:.4f}")
print(f"{'  Recall:':<40} {metrics['recall']:.4f}")
print(f"{'  F1 score 35,311 rows:':<40} {metrics['f1']:.4f}")
print(f"{'  F1 (Post-hoc F1 after excluding six one-token titles):':<40} {metrics_35305['f1']:.4f}")
print(f"\n{'  Paper F1 (p. 4811):':<40} {PAPER_F1}")
print(f"{'  Difference:':<40} {metrics['f1'] - PAPER_F1:+.4f} ({(metrics['f1'] - PAPER_F1) / PAPER_F1 * 100:+.1f}%)")
print("=" * 80)

RUNNING UPM ALGORITHM ON ORIGINAL (SORTED) DATA

Pipeline statistics:                    
  Products processed:                    35,311
  Initial clusters                       8,500
  Products evicted                       10,901
    -> reattached to existing:           1,653
    -> new clusters formed:              7,466
  Final clusters:                        16,400
  One-token titles:                      6
  Titles with no positive-score combination: 434 (incl. the one-token titles; kept as singletons)
  Mean tokens per title:                 8.479 (Table 3: 8.560)
  Runtime:                               116.5 seconds (paper: 11 s in C++, Table 5)

Evaluation results:                     
  Precision:                             0.6487
  Recall:                                0.5242
  F1 score 35,311 rows:                  0.5798
  F1 (Post-hoc F1 after excluding six one-token titles): 0.5799

  Paper F1 (p. 4811):                    0.588
  Difference:                        

### Experiment 2: shuffled rows

The CSV is sorted by ground-truth cluster, so any step that depends on row order could see the labels indirectly. Shuffled runs are the order-independent estimate.

In [9]:
print("=" * 80)
print("RUNNING UPM ON SHUFFLED DATA (3 RANDOM SEEDS)")
print("=" * 80)

shuffled_results, shuffled_stats, shuffled_dfs = [], [], []

for seed in range(1, 4):
    print(f"\nSeed {seed}:", end=" ")
    result_df_s, stats_s = run_upm(df, K=K, alpha=ALPHA, b=B, tau=TAU, shuffle_seed=seed)
    metrics_s = evaluate(result_df_s)
    shuffled_results.append(metrics_s)
    shuffled_stats.append(stats_s)
    shuffled_dfs.append(result_df_s)
    print(f"F1 = {metrics_s['f1']:.4f}, Clusters = {stats_s['final_clusters']:,}, Time = {stats_s['seconds']:.0f}s")

shuffled_f1s = [r["f1"] for r in shuffled_results]
mean_f1 = np.mean(shuffled_f1s)
std_f1 = np.std(shuffled_f1s, ddof=1)

print("\n" + "=" * 80)
print("SUMMARY: ROW ORDER EFFECT")
print("=" * 80)
print(f"{'Sorted data F1:':<40} {metrics['f1']:.4f}")
print(f"{'Shuffled data F1 (mean +/- std):':<40} {mean_f1:.4f} +/- {std_f1:.4f}")
print(f"{'Order effect (sorted - shuffled):':<40} {metrics['f1'] - mean_f1:+.4f}")
print(f"\n{'Paper F1':<40} {PAPER_F1}")
print(f"{'Gap (shuffled vs paper):':<40} {mean_f1 - PAPER_F1:+.4f} ({(mean_f1 - PAPER_F1) / PAPER_F1 * 100:+.1f}%)")
print("=" * 80)

RUNNING UPM ON SHUFFLED DATA (3 RANDOM SEEDS)

Seed 1: F1 = 0.5763, Clusters = 16,404, Time = 120s

Seed 2: F1 = 0.5743, Clusters = 16,413, Time = 128s

Seed 3: F1 = 0.5751, Clusters = 16,414, Time = 124s

SUMMARY: ROW ORDER EFFECT
Sorted data F1:                          0.5798
Shuffled data F1 (mean +/- std):         0.5752 +/- 0.0010
Order effect (sorted - shuffled):        +0.0046

Paper F1                                 0.588
Gap (shuffled vs paper):                 -0.0128 (-2.2%)


---
## Part 6: Verification

**Paper reference:** Eq. 7 (p. 4791): a cluster cannot contain multiple products from the same vendor.

In [10]:
# Equation 7 (p. 4791): no predicted cluster may contain two products of the same vendor.
# run_upm also asserts this internally; here it is checked again on the saved predictions of every run.
runs_to_check = {"sorted": result_df}
runs_to_check.update({f"shuffle seed {i + 1}": dfi for i, dfi in enumerate(shuffled_dfs)})

print("Vendor blocking constraint (p. 4791):")
for name, dfi in runs_to_check.items():
    counts = dfi.groupby(["Predicted Cluster", "Merchant ID"]).size()
    n_viol = int((counts > 1).sum())
    print(f"  {name:16s} clusters with >1 product of one vendor: {n_viol}   {'PASS' if n_viol == 0 else 'FAIL'}")

Vendor blocking constraint (p. 4791):
  sorted           clusters with >1 product of one vendor: 0   PASS
  shuffle seed 1   clusters with >1 product of one vendor: 0   PASS
  shuffle seed 2   clusters with >1 product of one vendor: 0   PASS
  shuffle seed 3   clusters with >1 product of one vendor: 0   PASS


---
## Part 7: Ablations

Each row changes one setting of the default configuration (strict `>`, dots/commas only between digits, hyphens and slashes split tokens, average length over occurrences). All runs use shuffled rows (seed 1). The noise of a single seed is about the shuffled standard deviation of Part 5.

In [11]:
# One change at a time, all on shuffled rows (seed 1), compared with the default run for seed 1.
# A single seed has noise of about the shuffled standard deviation (std_f1 above).
base_f1 = shuffled_results[0]["f1"]
ablations = {}

def ablate(name, **kw):
    d_, s_ = run_upm(df, K=K, alpha=ALPHA, b=B, tau=TAU, shuffle_seed=1, **kw)
    ablations[name] = {"f1": evaluate(d_)["f1"], "clusters": s_["final_clusters"], "mean_tokens": s_["mean_tokens"]}
    print(f"{name:52s} F1 = {ablations[name]['f1']:.4f}  (delta {ablations[name]['f1'] - base_f1:+.4f})  "
          f"mean tokens = {s_['mean_tokens']:.3f}", flush=True)

print(f"{'default (strict >, dots only between digits)':52s} F1 = {base_f1:.4f}")
ablate("tie '>=' instead of strict '>'", tie="ge")
ablate("average length over distinct combinations", avg_len="distinct")
ablate("dots/commas kept between any alphanumerics", sep="any")
ablate("hyphens and slashes kept inside tokens", keep_hyphen=True)

default (strict >, dots only between digits)         F1 = 0.5763
tie '>=' instead of strict '>'                       F1 = 0.5763  (delta -0.0000)  mean tokens = 8.479
average length over distinct combinations            F1 = 0.5763  (delta +0.0000)  mean tokens = 8.479
dots/commas kept between any alphanumerics           F1 = 0.5750  (delta -0.0013)  mean tokens = 8.474
hyphens and slashes kept inside tokens               F1 = 0.5457  (delta -0.0306)  mean tokens = 8.424


---
## Part 8: Summary

In [12]:
total_time = time.perf_counter() - NOTEBOOK_START

print("\n" + "=" * 80)
print(" " * 25 + "TASK 2 SUMMARY")
print("=" * 80)
print(f"\n{'Total notebook runtime:':<40} {total_time:.1f} seconds")
print(f"\n{'Paper F1 (p. 4811):':<40} {PAPER_F1}")
print(f"{'Reproduction F1, sorted file:':<40} {metrics['f1']:.4f}  ({metrics['f1'] - PAPER_F1:+.4f})")
print(f"{'Reproduction F1, shuffled rows:':<40} {mean_f1:.4f} +/- {std_f1:.4f}  ({mean_f1 - PAPER_F1:+.4f}, {(mean_f1 - PAPER_F1) / PAPER_F1 * 100:+.1f}%)")
print(f"{'Vendor constraint (Eq. 7):':<40} 0 violations in all runs (checked above)")
print("=" * 80)


                         TASK 2 SUMMARY

Total notebook runtime:                  1008.0 seconds

Paper F1 (p. 4811):                      0.588
Reproduction F1, sorted file:            0.5798  (-0.0082)
Reproduction F1, shuffled rows:          0.5752 +/- 0.0010  (-0.0128, -2.2%)
Vendor constraint (Eq. 7):               0 violations in all runs (checked above)
